# **VGG16 Model**



> Necessary libraries are imported



In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report
import pandas as pd
import os, re
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


> Connecting to Google Drive (ONLY if the file is being run on Google Collab)




In [2]:
from google.colab import drive

#Mounting Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


> Configure paths and constants before running any cells below.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
# Adjust these paths before running.
# When using Google Colab set DATA_ROOT inside your mounted Drive.

DATA_ROOT       = "/content/drive/MyDrive/Hanan Share/Data_augmentation_updated"
MODEL_SAVE_PATH = "/content/drive/MyDrive"  # directory for saved model files

IMG_SIZE    = 224    # height and width of each input image (pixels)
NORM_FACTOR = 40.0   # max LLS height value used for [0,1] normalisation
NUM_CLASSES = 4      # Infill=0, Solid=1, Wire Good=2, Wire Bad=3
TEST_SPLIT  = 0.2    # fraction of dataset held out for testing
RANDOM_SEED = 42


# **Loading Augmented Data**

In [ ]:
def load_class(class_subdir):
    """Return {filename_stem: DataFrame} for all CSVs in DATA_ROOT/class_subdir."""
    class_dir = os.path.join(DATA_ROOT, class_subdir)
    files = [f for f in os.listdir(class_dir) if f.endswith('.csv')]
    return {
        os.path.splitext(f)[0]: pd.read_csv(os.path.join(class_dir, f), header=None)
        for f in files
    }


In [ ]:
Infill_df    = load_class("Infill")
Solid_df     = load_class("Solid")
Wire_good_df = load_class("Wire_good")
Wire_bad_df  = load_class("Wire_bad")

print(f"Infill: {len(Infill_df)}, Solid: {len(Solid_df)}, "
      f"Wire Good: {len(Wire_good_df)}, Wire Bad: {len(Wire_bad_df)}")


# **Concatenating all Data Frames and Assigning Labels**

In [ ]:
concatenate_data_dic = {**Infill_df, **Solid_df, **Wire_good_df, **Wire_bad_df}

# Trim any off-by-one extra row/col produced during augmentation
for key in concatenate_data_dic:
    df = concatenate_data_dic[key]
    if df.shape[0] == IMG_SIZE + 1:
        concatenate_data_dic[key] = df.iloc[:-1, :]
    if df.shape[1] == IMG_SIZE + 1:
        concatenate_data_dic[key] = concatenate_data_dic[key].iloc[:, :-1]

data_values = {k: v.values for k, v in concatenate_data_dic.items()}

# Numeric labels: 0=Infill, 1=Solid, 2=Wire Good, 3=Wire Bad
labels = np.array(
    [0] * len(Infill_df) +
    [1] * len(Solid_df) +
    [2] * len(Wire_good_df) +
    [3] * len(Wire_bad_df)
)


# **Reshaping and Normalization (pseudo-RGB for VGG16)**

In [ ]:
data_list = []
for values in data_values.values():
    gray = values.reshape(IMG_SIZE, IMG_SIZE, 1)
    rgb  = np.repeat(gray, 3, axis=-1).astype(np.float32) / NORM_FACTOR
    data_list.append(rgb)

data_vgg16 = np.array(data_list)
print(f"VGG16 input shape: {data_vgg16.shape}")


# **Splitting into Training and Testing Sets and Importing Base VGG16 Model**

In [ ]:
X_train_vgg16, X_test_vgg16, y_train_vgg16, y_test_vgg16 = train_test_split(
    data_vgg16, labels, test_size=TEST_SPLIT, random_state=RANDOM_SEED
)

base_model_vgg16 = VGG16(weights='imagenet', include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))


# **Adding a Top Layer and Manipulation of VGG16 layers**

In [ ]:
# Unfreeze the last 10 layers for fine-tuning
for layer in base_model_vgg16.layers[-10:]:
    layer.trainable = True

model_vgg16 = Sequential([
    base_model_vgg16,
    layers.Conv2D(32, (3, 3), activation='PReLU', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='PReLU', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='PReLU', padding='same'),
    layers.Flatten(),
    layers.Dense(64, activation='PReLU'),
    layers.Dense(NUM_CLASSES, activation='softmax'),
])

# Freeze VGG16 base layers
for layer in base_model_vgg16.layers:
    layer.trainable = False

model_vgg16.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stopping_vgg16 = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_vgg16 = model_vgg16.fit(
    X_train_vgg16, y_train_vgg16,
    epochs=50,
    validation_split=TEST_SPLIT,
    callbacks=[early_stopping_vgg16],
)


# **Evaluation of VGG16 Model**

In [ ]:
model_vgg16.evaluate(X_test_vgg16, y_test_vgg16)

y_pred_classes_vgg16 = np.argmax(model_vgg16.predict(X_test_vgg16), axis=1)


# **Classification Report**

In [ ]:
CLASS_NAMES = ['Infill', 'Solid', 'Wire_Good', 'Wire_Bad']


In [ ]:
print(classification_report(y_test_vgg16, y_pred_classes_vgg16, target_names=CLASS_NAMES))


# **Confusion Matrix**

In [ ]:
conf_matrix = confusion_matrix(y_test_vgg16, y_pred_classes_vgg16)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix - VGG16')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.show()


# **Loss and Accuracy Plots**

In [ ]:
plt.plot(history_vgg16.history['accuracy'],     label='Train Accuracy')
plt.plot(history_vgg16.history['val_accuracy'], label='Validation Accuracy')
plt.title('VGG16 Model Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()
plt.show()

plt.plot(history_vgg16.history['loss'],     label='Train Loss')
plt.plot(history_vgg16.history['val_loss'], label='Validation Loss')
plt.title('VGG16 Model Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
plt.show()


# **Save Model**

In [ ]:
save_path = os.path.join(MODEL_SAVE_PATH, 'VGG16CNN.keras')
model_vgg16.save(save_path)
print(f'Model saved to {save_path}')
